In [24]:
import seaborn as sns

taxi = sns.load_dataset('taxis')
taxi.info()
subtaxi = taxi.loc[:, ['passengers', 'distance', 'fare', 'tip', 'tolls', 'total']]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6433 entries, 0 to 6432
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   pickup           6433 non-null   datetime64[ns]
 1   dropoff          6433 non-null   datetime64[ns]
 2   passengers       6433 non-null   int64         
 3   distance         6433 non-null   float64       
 4   fare             6433 non-null   float64       
 5   tip              6433 non-null   float64       
 6   tolls            6433 non-null   float64       
 7   total            6433 non-null   float64       
 8   color            6433 non-null   object        
 9   payment          6389 non-null   object        
 10  pickup_zone      6407 non-null   object        
 11  dropoff_zone     6388 non-null   object        
 12  pickup_borough   6407 non-null   object        
 13  dropoff_borough  6388 non-null   object        
dtypes: datetime64[ns](2), float64(5), int64(

In [26]:
import numpy as np
# create missingness
np.random.seed(42)
mask1 = subtaxi.total > 25
mask2 = np.random.rand(subtaxi.shape[0]) < 0.5

subtaxi_missing = subtaxi.copy()
subtaxi_missing.loc[mask1 & mask2, 'tip'] = np.nan

In [28]:
## Mean Simple Imputation

from sklearn.impute import SimpleImputer

mean_imputer = SimpleImputer(strategy='mean')

taxis_mean = subtaxi_missing.copy()
taxis_mean['tip'] = mean_imputer.fit_transform(taxis_mean[['tip']])

In [30]:
## Median Simple Imputation

median_imputer = SimpleImputer(strategy='median')

taxis_median = subtaxi_missing.copy()
taxis_median['tip'] = median_imputer.fit_transform(taxis_median[['tip']])

In [37]:
## K Nearest Neighbor

import pandas as pd
from sklearn.impute import KNNImputer

# keep only numeric columns (same as your prof)
taxis_knn = subtaxi_missing.select_dtypes('number').copy()

# apply KNN
knn_imputer = KNNImputer(n_neighbors=3)
taxis_knn_impute = knn_imputer.fit_transform(taxis_knn)

# convert back to dataframe
taxis_knn_df = pd.DataFrame(taxis_knn_impute, columns=taxis_knn.columns)

In [34]:
## Predictive Mean Matching 

import miceforest as mf

kernel_pmm = mf.ImputationKernel(
    data=subtaxi_missing.select_dtypes('number'),
    num_datasets=1,
    mean_match_candidates=5,
    random_state=1,
    save_all_iterations_data=False
)

kernel_pmm.mice(iterations=15)
taxis_pmm = kernel_pmm.complete_data()

In [41]:
from sklearn.metrics import mean_squared_error
import numpy as np

missing_mask = mask1 & mask2

rmse_mean = np.sqrt(mean_squared_error(subtaxi.loc[missing_mask, 'tip'], taxis_mean.loc[missing_mask, 'tip']))

rmse_median = np.sqrt(mean_squared_error(subtaxi.loc[missing_mask, 'tip'], taxis_median.loc[missing_mask, 'tip']))

rmse_knn = np.sqrt(mean_squared_error(subtaxi.loc[missing_mask, 'tip'], taxis_knn_df.loc[missing_mask, 'tip']))

rmse_pmm = np.sqrt(mean_squared_error(subtaxi.loc[missing_mask, 'tip'], taxis_pmm.loc[missing_mask, 'tip']))

print("Mean RMSE:", rmse_mean)
print("Median RMSE:", rmse_median)
print("KNN RMSE:", rmse_knn)
print("PMM RMSE:", rmse_pmm)

Mean RMSE: 5.157972738822796
Median RMSE: 5.214790511331332
KNN RMSE: 1.9158276234478935
PMM RMSE: 2.411911912807868


In t